# Phase 1 — Training Sweep

Runs all **7 ratios × 3 seeds = 21 training runs** (≈ 4 h on T4 at 6 000 epochs with the pooled dataset).

**Kaggle:** Enable **GPU T4** and **Internet** in the right panel, then *Save and Run All*.
Checkpoints land in `/kaggle/working/slt/results/` and are saved automatically as output.

> ⚠️ **Do NOT add any previous Phase 1 output as dataset input** — the seed bug AND the data-construction bug were fixed; old checkpoints are invalid. Start fresh with no dataset inputs.

**Next step after this finishes:** open `02_calibrate.ipynb`,
add this notebook's output version as a dataset input, and run it.

## Section 0 — Setup

In [ ]:
import os, sys, shutil, subprocess, glob

PLATFORM = "kaggle"   # "kaggle" or "colab"
REPO_URL  = "https://github.com/makataomu/slt-diplomka"

if PLATFORM == "colab":
    from google.colab import drive; drive.mount("/content/drive")
    REPO_DIR    = "/content/slt"
    PERSIST_DIR = "/content/drive/MyDrive/slt_persist"
    for d in ["results/checkpoints", "results/metrics", "results/figures"]:
        os.makedirs(f"{PERSIST_DIR}/{d}", exist_ok=True)
else:
    REPO_DIR    = "/kaggle/working/slt"
    PERSIST_DIR = None

if os.path.exists(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    print("Pulled latest from GitHub")
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("Cloned from GitHub")

if PLATFORM == "colab":
    lnk = f"{REPO_DIR}/results"
    if os.path.islink(lnk): os.unlink(lnk)
    elif os.path.isdir(lnk): shutil.rmtree(lnk)
    os.symlink(f"{PERSIST_DIR}/results", lnk)
    print(f"results/ -> {PERSIST_DIR}/results")
else:
    for sub in ["results/checkpoints", "results/metrics", "results/figures"]:
        os.makedirs(f"{REPO_DIR}/{sub}", exist_ok=True)
    # Resume support: restore any missing checkpoints/metrics from a previous
    # 01_train output added as dataset input. Handles the nested Kaggle path
    # /kaggle/input/notebooks/{user}/{name}/slt/results/
    for prev in glob.glob("/kaggle/input/**/slt/results", recursive=True):
        print(f"Restoring from {prev} ...")
        for sub in ["checkpoints", "metrics"]:
            src, dst = f"{prev}/{sub}", f"{REPO_DIR}/results/{sub}"
            if os.path.exists(src):
                for item in os.listdir(src):
                    s, d = f"{src}/{item}", f"{dst}/{item}"
                    if not os.path.exists(d):
                        (shutil.copytree if os.path.isdir(s) else shutil.copy2)(s, d)
        print("  Done.")

os.chdir(REPO_DIR)
sys.path.insert(0, f"{REPO_DIR}/src")
print(f"\nReady. Platform={PLATFORM} | cwd={os.getcwd()}")


## Section 1 — Install dependencies

> **Kaggle:** Internet must be On in the right panel.

In [ ]:
%pip install -q transformer_lens devinterp zarr==3.1.6

import importlib.metadata, torch
print(f"devinterp {importlib.metadata.version('devinterp')}  "
      f"| torch {torch.__version__}  "
      f"| device: {'cuda' if torch.cuda.is_available() else 'cpu'}")


## Section 2 — Full training sweep

7 ratios × 3 seeds, 6 000 epochs each, checkpoint every 60 epochs (matches Sullivan's protocol).
`--resume` is always on so a partial run can be safely continued.

In [ ]:
RATIOS = [0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.70]
SEEDS  = [0, 1, 2]
EPOCHS           = 6_000
CHECKPOINT_EVERY = 60

for ratio in RATIOS:
    for seed in SEEDS:
        print(f"\n{'='*60}")
        print(f"ratio={ratio}  seed={seed}")
        print(f"{'='*60}")
        !python src/train.py --ratio {ratio} --seed {seed} \
            --epochs {EPOCHS} --checkpoint_every {CHECKPOINT_EVERY} --resume

print("\nAll training runs complete.")


## Section 3 — Diagnostics

In [ ]:
EPOCHS = 6_000

from pathlib import Path
import pandas as pd

print("=== Checkpoints ===")
for rd in sorted(Path("results/checkpoints").glob("ratio_*")):
    for sd in sorted(rd.glob("seed_*")):
        ckpts = sorted(sd.glob("epoch_*.pt"))
        if ckpts:
            last = int(ckpts[-1].stem.split("_")[1])
            done = last == EPOCHS
            print(f"  {rd.name}/{sd.name}: {len(ckpts)} ckpts, "
                  f"last={last} {'✓' if done else '(incomplete)'}")

print("\n=== Training metrics ===")
for f in sorted(Path("results/metrics").glob("ratio_*.csv")):
    if "_llc" not in f.name:
        df = pd.read_csv(f)
        print(f"  {f.name}: {len(df)} rows, max_epoch={df['epoch'].max()}")
